In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("anime.csv")

In [3]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
df.shape

(12294, 7)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [6]:
df.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df["genre"] = df["genre"].fillna("Unknown")

df["type"] = df["type"].fillna(df["type"].mode()[0])

df["rating"] = df["rating"].fillna(df["rating"].mean())

In [9]:
df.isnull().sum()

,0
anime_id,0
name,0
genre,0
type,0
episodes,0
rating,0
members,0


In [25]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()

genre_matrix = cv.fit_transform(df["genre"])

In [12]:
genre_df = pd.DataFrame(
    genre_matrix.toarray(),
    columns=cv.get_feature_names_out()
)

genre_df.head()

,action,adventure,ai,arts,cars,comedy,dementia,demons,drama,ecchi,...,slice,space,sports,super,supernatural,thriller,unknown,vampire,yaoi,yuri
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
1,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_features = scaler.fit_transform(
    df[["rating","members"]]
)

In [14]:
from scipy.sparse import hstack

feature_matrix = hstack(
    [genre_matrix, num_features]
)

feature_matrix.shape

(12294, 50)

In [15]:
# Calculating similarity between all anime

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

In [16]:
# Checking similarity matrix shape

similarity_matrix.shape

(12294, 12294)

In [17]:
# Function to recommend similar anime

def recommend_anime(anime_name):

    index = df[df["name"] == anime_name].index[0]

    similarity_scores = list(
        enumerate(similarity_matrix[index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    top_anime = similarity_scores[1:6]

    for i in top_anime:
        print(
            df.iloc[i[0]]["name"],
            "-> Similarity:",
            round(i[1], 3)
        )

In [18]:
# Testing recommendation system

recommend_anime("Naruto")

Naruto: Shippuuden -> Similarity: 0.997
Bleach -> Similarity: 0.989
Shingeki no Kyojin -> Similarity: 0.986
Kill la Kill -> Similarity: 0.98
Angel Beats! -> Similarity: 0.974


In [21]:
# Recommendation using threshold value

def recommend_threshold(anime_name, threshold=0.95):

    index = df[df["name"] == anime_name].index[0]

    similarity_scores = list(
        enumerate(similarity_matrix[index])
    )

    for i, score in similarity_scores:

        if score >= threshold and i != index:

            print(
                df.iloc[i]["name"],
                "->",
                round(score,3)
            )

In [22]:
# Recommendations above 0.95 similarity

recommend_threshold(
    "Naruto",
    threshold=0.95
)

Fullmetal Alchemist: Brotherhood -> 0.969
Steins;Gate -> 0.962
Hunter x Hunter (2011) -> 0.955
Code Geass: Hangyaku no Lelouch R2 -> 0.957
Code Geass: Hangyaku no Lelouch -> 0.97
One Punch Man -> 0.96
Tengen Toppa Gurren Lagann -> 0.959
Death Note -> 0.97
One Piece -> 0.969
Shingeki no Kyojin -> 0.986
Fate/Zero -> 0.957
Psycho-Pass -> 0.952
No Game No Life -> 0.958
Toradora! -> 0.96
Angel Beats! -> 0.974
Durarara!! -> 0.968
Fullmetal Alchemist -> 0.965
Dragon Ball Z -> 0.957
Darker than Black: Kuro no Keiyakusha -> 0.956
Kill la Kill -> 0.98
Fairy Tail -> 0.972
D.Gray-man -> 0.951
Noragami -> 0.969
Soul Eater -> 0.973
Mirai Nikki (TV) -> 0.97
Tokyo Ghoul -> 0.957
Hataraku Maou-sama! -> 0.95
Bleach -> 0.989
Naruto: Shippuuden -> 0.997
Ao no Exorcist -> 0.97
Another -> 0.951
Elfen Lied -> 0.958
Akame ga Kill! -> 0.967
Sword Art Online -> 0.973
Guilty Crown -> 0.963
Deadman Wonderland -> 0.958
Highschool of the Dead -> 0.965
Sword Art Online II -> 0.959


In [23]:
# Recommendations above 0.90 similarity

recommend_threshold(
    "Naruto",
    threshold=0.90
)

Fullmetal Alchemist: Brotherhood -> 0.969
Steins;Gate -> 0.962
Hunter x Hunter (2011) -> 0.955
Clannad: After Story -> 0.914
Code Geass: Hangyaku no Lelouch R2 -> 0.957
Sen to Chihiro no Kamikakushi -> 0.942
Shigatsu wa Kimi no Uso -> 0.927
Code Geass: Hangyaku no Lelouch -> 0.97
Cowboy Bebop -> 0.942
One Punch Man -> 0.96
Mononoke Hime -> 0.922
Tengen Toppa Gurren Lagann -> 0.959
Fate/Zero 2nd Season -> 0.914
Death Note -> 0.97
Boku dake ga Inai Machi -> 0.926
Re:Zero kara Hajimeru Isekai Seikatsu -> 0.911
Ano Hi Mita Hana no Namae wo Bokutachi wa Mada Shiranai. -> 0.934
Shokugeki no Souma -> 0.931
Kiseijuu: Sei no Kakuritsu -> 0.922
One Piece -> 0.969
Baccano! -> 0.92
Shingeki no Kyojin -> 0.986
Fate/Zero -> 0.957
Mahou Shoujo Madoka★Magica -> 0.942
Psycho-Pass -> 0.952
Samurai Champloo -> 0.941
Noragami Aragoto -> 0.915
No Game No Life -> 0.958
Kuroko no Basket -> 0.933
Toradora! -> 0.96
Nanatsu no Taizai -> 0.909
Sakurasou no Pet na Kanojo -> 0.91
Angel Beats! -> 0.974
Bakemonogata

In [24]:
# Recommendations above 0.85 similarity

recommend_threshold(
    "Naruto",
    threshold=0.85
)

Fullmetal Alchemist: Brotherhood -> 0.969
Steins;Gate -> 0.962
Hunter x Hunter (2011) -> 0.955
Clannad: After Story -> 0.914
Gintama -> 0.887
Code Geass: Hangyaku no Lelouch R2 -> 0.957
Sen to Chihiro no Kamikakushi -> 0.942
Shigatsu wa Kimi no Uso -> 0.927
Code Geass: Hangyaku no Lelouch -> 0.97
Cowboy Bebop -> 0.942
One Punch Man -> 0.96
Mononoke Hime -> 0.922
Tengen Toppa Gurren Lagann -> 0.959
Howl no Ugoku Shiro -> 0.898
Fate/Zero 2nd Season -> 0.914
Death Note -> 0.97
Haikyuu!! -> 0.885
Boku dake ga Inai Machi -> 0.926
Re:Zero kara Hajimeru Isekai Seikatsu -> 0.911
Ano Hi Mita Hana no Namae wo Bokutachi wa Mada Shiranai. -> 0.934
Shokugeki no Souma -> 0.931
Hellsing Ultimate -> 0.871
Kiseijuu: Sei no Kakuritsu -> 0.922
Kuroko no Basket 2nd Season -> 0.871
One Piece -> 0.969
Baccano! -> 0.92
Shingeki no Kyojin -> 0.986
Shinsekai yori -> 0.853
Fate/Zero -> 0.957
Mahou Shoujo Madoka★Magica -> 0.942
Nichijou -> 0.852
Psycho-Pass -> 0.952
Magi: The Kingdom of Magic -> 0.86
Samurai Cha

**Performance Analysis**

The recommendation system was created using cosine similarity.

It was able to recommend anime that are similar to the selected anime based on genre, rating, and number of members.

When the similarity threshold was increased, fewer recommendations were generated. When the threshold was decreased, more recommendations were generated.

The recommendation system worked well and provided relevant anime suggestions.

The results can be improved further by using user ratings and collaborative filtering methods.


Q1. Difference between User-Based and Item-Based Collaborative Filtering


User-Based Collaborative Filtering recommends items based on the preferences of similar users.

Item-Based Collaborative Filtering recommends items that are similar to the items already liked by the user.

In simple terms, User-Based focuses on similar users, while Item-Based focuses on similar items.


Q2. What is Collaborative Filtering and How Does It Work?



Collaborative Filtering is a recommendation technique used to suggest items to users.

It works by finding similarities between users or items and then recommending items based on those similarities.

For example, if two users have similar interests, items liked by one user can be recommended to the other user.
